# Agent 测试 Notebook

这个 notebook 用来测试 AlpineFlow AI Agent 系统。

## 0. 环境设置 + 对话测试 (必须先运行)

In [ ]:
import sys
from pathlib import Path

# 设置路径
ROOT = Path.cwd()
if ROOT.name == "agent":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 测试 chat 功能
from agent import chat

print("环境设置完成!")
print()
print("=" * 50)
print("测试对话:")
print("=" * 50)
print(chat("明天去萨尔茨堡怎么样"))

## 1. 更多对话示例

In [ ]:
# 通勤者
print(chat("我每天上班通勤，今天下班几点走最好"))

In [ ]:
# 周末出游
print(chat("周末想带家人去萨尔茨堡玩"))

In [ ]:
# 物流司机
print(chat("我是货车司机，后天要送货去因斯布鲁克"))

In [ ]:
# 暑假规划
print(chat("暑假想去奥地利自驾游，什么时候最好"))

---

## 2. 详细组件测试 (开发调试用)

以下测试各个 Agent 组件，需要先运行此 cell 进行初始化。

In [ ]:
import asyncio
import json
from dataclasses import asdict, is_dataclass
from typing import Any

from agent.models import AgentRequest, UserType
from agent.agents import (
    IntentParser,
    ForecastAgent,
    ContextAgent,
    SearchAgent,
    GenerationAgent,
)
from agent.orchestrator import Orchestrator
from agent.personas import PersonaType

# 测试常量
TEST_QUERY = "我住在萨尔茨堡附近，每周日早上都会去慕尼黑，给我往返的推荐时间"
TEST_DATE = "2026-07-25"
TEST_ROAD = "A8"
TEST_DESTINATION = "salzburg"
TEST_HOURS = [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


def to_jsonable(value: Any) -> Any:
    if is_dataclass(value):
        return to_jsonable(asdict(value))
    if isinstance(value, dict):
        return {key: to_jsonable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_jsonable(item) for item in value]
    if hasattr(value, "value"):
        return value.value
    return value


def show(title: str, value: Any) -> None:
    print(f"\n{'=' * 10} {title} {'=' * 10}")
    if isinstance(value, str):
        print(value)
    else:
        print(json.dumps(to_jsonable(value), ensure_ascii=False, indent=2))


def show_response(title: str, response) -> None:
    """显示 AgentResponse 结果"""
    print(f"\n{'=' * 10} {title} {'=' * 10}")
    print(json.dumps(to_jsonable({
        "success": response.success,
        "data": response.data,
    }), ensure_ascii=False, indent=2))


print("详细测试环境准备完成!")
print(f"TEST_DATE: {TEST_DATE}")
print(f"TEST_ROAD: {TEST_ROAD}")
print(f"TEST_DESTINATION: {TEST_DESTINATION}")

### 2.1 IntentParser 测试

In [ ]:
intent_parser = IntentParser(use_llm=True)
parsed_intent = await intent_parser.parse_async(TEST_QUERY, UserType.TRAVELER)

show("IntentParser", {
    "persona_type": parsed_intent.persona_type,
    "user_type": parsed_intent.user_type,
    "core_question": parsed_intent.core_question,
    "destination": parsed_intent.destination,
    "road": parsed_intent.road,
    "intent": parsed_intent.intent,
    "time_range": parsed_intent.time_range,
    "trip_plan": parsed_intent.trip_plan,
})

### 2.2 ForecastAgent 测试

In [ ]:
forecast_agent = ForecastAgent()

hourly_request = AgentRequest(
    query="forecast hourly test",
    date=TEST_DATE,
    road=TEST_ROAD,
    destination=TEST_DESTINATION,
    hours=TEST_HOURS,
    granularity="hourly",
)
hourly_forecast_result = await forecast_agent.process(hourly_request)
show_response("ForecastAgent hourly", hourly_forecast_result)

### 2.3 ContextAgent 测试

In [ ]:
context_agent = ContextAgent()
context_request = AgentRequest(
    query="context test",
    date=TEST_DATE,
    road=TEST_ROAD,
    destination=TEST_DESTINATION,
    start_date="2026-07-25",
    end_date="2026-07-27",
    hours=[7, 8, 9],
)
context_result = await context_agent.process(context_request)

show("ContextAgent", {
    "success": context_result.success,
    "factors_count": len(context_result.data.get("factors", [])),
})

if context_result.success:
    show("Context summary", context_result.data.get("summary"))

### 2.4 SearchAgent 测试

In [ ]:
search_agent = SearchAgent()
search_request = AgentRequest(
    query="search real-time factors test",
    date=TEST_DATE,
    road=TEST_ROAD,
    destination=TEST_DESTINATION,
)
search_result = await search_agent.process(search_request)

show("SearchAgent", {
    "success": search_result.success,
    "factors_count": len(search_result.data.get("factors", [])),
})

if search_result.success:
    for f in search_result.data.get("factors", []):
        print(f"  - [{f.type}] {f.name}: {f.description[:50]}...")

### 2.5 完整链路测试 (Orchestrator)

In [ ]:
from agent import ask

# 带调试信息的完整测试
result = ask("我想在 2026-07-25 到 2026-07-31 去 Salzburg，哪天出发最好？", verbose=True)
print()
print("=" * 50)
print("最终建议:")
print("=" * 50)
print(result)

### 2.6 LLM 连接测试

In [ ]:
from agent import config
from agent.tools import LLMClient

show("LLM Config", {
    "provider": config.LLM_PROVIDER,
    "model": config.OPENAI_MODEL,
    "key_configured": bool(config.OPENAI_API_KEY),
})

llm_client = LLMClient()
llm_text = await llm_client.generate(
    "Reply with exactly: GPT_OK",
    system="You are a minimal API connectivity test.",
    temperature=0,
    max_tokens=20,
)

show("LLM Test", {
    "success": "GPT_OK" in llm_text,
    "response": llm_text.strip(),
})

---

## 3. 自定义测试

在下面的 cell 中输入你自己的问题进行测试。

In [ ]:
# 在这里输入你的问题
my_question = "下周五去萨尔茨堡，几点出发最好？"

print(chat(my_question))